# Revision analyses 

One section per analysis. Every section writes its
tables, figure and a `numbers.json` to `results/revision/<section>/` so the Supplement can
be filled from files rather than from notebook output.

| # | Section | Status |
|---|---|---|
| 1 | Cohort analysis: stayers, entrants, leavers; within-politician change; politician fixed effects | ported from `code/04_analysis/0405_composition_stayers.py` |
| 2 | Pre-2015 extension: EMI for all Bundestag speeches 2000–2025, jointly standardised; shifts at all seven elections, placebo breaks, single-break search | done |
| 2b | 2017 sparse-quarter check and alternative breakpoint date | planned |
| 3 | Grouped lexicon ablation (leave-one-group-out, core-only) | planned |
| 4 | Self- vs other-attribution coding | planned |
| 5 | Discriminant validity (moralisation, emotionality, populism, incivility) | planned |
| 6 | Hyperparameter multiverse (dims × window × seeds) | planned |
| 7 | Document-level validation AUC by party and period (eras, decades, parties; attention-check exclusion applied) | done |
| 8 | Topic controls on identical samples, noise cluster kept as its own category; exclusion rates by party and period | parliament done, Twitter via `code/05_validation/topics/0542_topic_twitter_same_sample.py` |
| 9 | Mechanism test at 2017: within-MdB EMI change vs ideological distance to the AfD (Breunig & Guinaudeau 2025) and constituency AfD exposure | done |
| 10 | Mechanism battery: 2021 transition, adoption hazard, long-run drift, similarity to AfD language, debate contact, wings, 2013 placebo, lead-lag | done (`code/04_analysis/mechanisms/0431_mechanism_tests.py`) |
| 12 | Within-sitting reaction to AfD speeches: local before/after windows, placebo parties, position and climate controls, replies that mention the AfD, share of the 2017 shift due to confrontation | done (`code/04_analysis/contagion/0415_afd_sequence_effect.py`, `code/04_analysis/contagion/0417_afd_local_window.py`, `code/04_analysis/contagion/0418_afd_local_window_checks.py`, `code/04_analysis/contagion/0419_afd_confrontation.py`) |


## 0. Setup

In [ ]:
import os
if not os.path.isdir('code'): os.chdir(os.path.join(os.getcwd(), '..', '..'))   # run from code/04_analysis: move to the repository root
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM
warnings.filterwarnings('ignore')

HERE = os.getcwd()                        # repository root (see first cell)
ROOT = HERE
DATA = os.path.join(HERE, 'data')         # shared corpora
RES  = os.path.join(HERE, 'results', 'revision'); os.makedirs(RES, exist_ok=True)   # results/revision/<section>/

EMI = 'emi_w2vparliament'
BP = {'elec2017': pd.Timestamp('2017-07-01', tz='UTC'),
      'elec2021': pd.Timestamp('2021-07-01', tz='UTC'),
      'elec2025': pd.Timestamp('2025-01-01', tz='UTC')}
COLORS = {'left': '#E66101', 'middle': '#4D4D4D', 'right': '#0072B2'}
LAB = {'left': 'Left', 'middle': 'Centre', 'right': 'Right'}
PARTY_COL = {'CDU/CSU': '#000000', 'SPD': '#E3000F', 'FDP': '#FFED00', 'Alliance 90/The Greens': '#46962B',
             'The Left Party': '#BE3075', 'AFD': '#009EE0'}

plt.rcParams.update({'font.size': 13, 'axes.labelsize': 15, 'legend.fontsize': 12, 'figure.facecolor': 'white',
                     'axes.facecolor': 'white', 'text.color': 'black', 'axes.labelcolor': 'black',
                     'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black'})

def boot_ci(x, n=2000, seed=0):
    r = np.random.default_rng(seed); x = np.asarray(x)
    m = [r.choice(x, len(x)).mean() for _ in range(n)]
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def save_numbers(num, outdir):
    json.dump(num, open(os.path.join(outdir, 'numbers.json'), 'w'), indent=1, default=float)
    print('saved', os.path.join(outdir, 'numbers.json'))

In [ ]:
# Parliament corpus (speech level). Only the columns needed here.
par = pd.read_csv(os.path.join(DATA, 'parliament_nhb.csv'),
                  usecols=['politicianId', 'date', 'electoralTerm', 'party', 'leaning', EMI, 'firstName', 'lastName'])
par = par.dropna(subset=['politicianId', 'party', 'leaning', EMI, 'date'])
par['politicianId'] = par['politicianId'].astype(int)
par['date'] = pd.to_datetime(par['date'], utc=True)
par['time_quarterly'] = (par['date'].dt.year * 10 + par['date'].dt.quarter).astype(int)
par['qdate'] = par['date'].dt.tz_localize(None).dt.to_period('Q').dt.to_timestamp()
par['year'] = par['date'].dt.year
par['late'] = (par['date'] >= '2020-01-01').astype(int)     # early = 2015-2019, late = 2020-2025 (as in the paper)
print(len(par), 'speeches;', par.politicianId.nunique(), 'politicians;', par.date.min().date(), '-', par.date.max().date())

## 1. Cohort analysis: is the parliamentary decline compositional or within-person?

Does the stepwise decline reflect a change in *who* speaks
(new members, the AfD, the BSW) or a change in *how* continuing politicians speak. Four pieces of evidence:

1. **Cohorts.** Politicians observed in both periods ("stayers"), only early ("leavers"), only late ("entrants"); two
   balanced panels, A = speak in WP18, WP19 and WP20 (no AfD possible), B = speak in WP19 and WP20 (includes AfD).
2. **Early/late comparison** on every sample and camp.
3. **Shift-share decomposition** of the raw early-to-late change into a within-stayer and a compositional component.
4. **Within-politician paired change** and the paper's breakpoint models re-estimated on the panels, with random
   intercepts and with politician fixed effects (within estimator).

Ported from `code/04_analysis/0405_composition_stayers.py` (results previously in `results/parliament/stayers/`, 26 Aug 2026).

In [ ]:
OUT1 = os.path.join(RES, '01_cohort_stayers'); os.makedirs(OUT1, exist_ok=True)
NUM = {}
df = par.copy()
NUM['n_speeches_all'] = int(len(df)); NUM['n_pol_all'] = int(df.politicianId.nunique())

# --- 1.1 cohorts -------------------------------------------------------------------------------
terms = df.groupby('politicianId')['electoralTerm'].agg(lambda s: set(s))
panelA = set(terms[terms.apply(lambda s: {18, 19, 20} <= s)].index)
panelB = set(terms[terms.apply(lambda s: {19, 20} <= s)].index)
pres = df.groupby('politicianId')['late'].agg(['min', 'max'])
stayers   = set(pres[(pres['min'] == 0) & (pres['max'] == 1)].index)
leavers   = set(pres[pres['max'] == 0].index)
entrants  = set(pres[pres['min'] == 1].index)
df['cohort'] = np.select([df.politicianId.isin(stayers), df.politicianId.isin(leavers)], ['stayer', 'leaver'], 'entrant')

NUM.update(n_pol_stayers=len(stayers), n_pol_leavers=len(leavers), n_pol_entrants=len(entrants),
           n_speeches_entrants=int(df.politicianId.isin(entrants).sum()),
           n_speeches_leavers=int(df.politicianId.isin(leavers).sum()),
           share_late_speeches_by_entrants=float(df[df.late == 1].politicianId.isin(entrants).mean()),
           share_early_speeches_by_leavers=float(df[df.late == 0].politicianId.isin(leavers).mean()))
coh = df.groupby(['cohort', 'party']).agg(n_pol=('politicianId', 'nunique'), n_speeches=(EMI, 'size'),
                                          mean_emi=(EMI, 'mean')).round(3)
coh.to_csv(os.path.join(OUT1, 'cohorts_by_party.csv'))
print({k: v for k, v in NUM.items() if k.startswith('n_pol') or k.startswith('share')})
coh

In [ ]:
# --- 1.2 early/late comparison by sample and camp --------------------------------------------
def early_late(d):
    e, l = d.loc[d.late == 0, EMI].values, d.loc[d.late == 1, EMI].values
    t, p = stats.ttest_ind(e, l, equal_var=False)
    sp = np.sqrt(((len(e) - 1) * e.var(ddof=1) + (len(l) - 1) * l.var(ddof=1)) / (len(e) + len(l) - 2))
    return dict(n_early=len(e), n_late=len(l), mean_early=e.mean(), mean_late=l.mean(),
                ci_early=boot_ci(e), ci_late=boot_ci(l), t=t, p=p, d=(l.mean() - e.mean()) / sp,
                mw_p=stats.mannwhitneyu(e, l).pvalue)

SAMPLES = [('all', None), ('stayers', stayers), ('panelA_WP18-20', panelA), ('panelB_WP19-20', panelB)]
rows = []
for name, ids in SAMPLES:
    d = df if ids is None else df[df.politicianId.isin(ids)]
    for lean in ['all', 'middle', 'left', 'right']:
        dd = d if lean == 'all' else d[d.leaning == lean]
        if dd.late.nunique() < 2 or (dd.late == 0).sum() < 30:
            continue
        r = early_late(dd); r.update(sample=name, leaning=lean, n_pol=dd.politicianId.nunique()); rows.append(r)
el = pd.DataFrame(rows); el.to_csv(os.path.join(OUT1, 'early_late_by_sample.csv'), index=False)
el[['sample', 'leaning', 'n_pol', 'n_early', 'n_late', 'mean_early', 'mean_late', 'd', 'p']].round(3)

In [ ]:
# --- 1.3 shift-share decomposition of the raw early -> late change ---------------------------
# late_mean - early_mean = within-stayer change (at average stayer weight) + composition (entrants in, leavers out)
e, l = df[df.late == 0], df[df.late == 1]
w_s_e, w_s_l = e.politicianId.isin(stayers).mean(), l.politicianId.isin(stayers).mean()
m_s_e, m_s_l = e.loc[e.politicianId.isin(stayers), EMI].mean(), l.loc[l.politicianId.isin(stayers), EMI].mean()
m_lv, m_en = e.loc[e.politicianId.isin(leavers), EMI].mean(), l.loc[l.politicianId.isin(entrants), EMI].mean()
total = l[EMI].mean() - e[EMI].mean()
w_bar = (w_s_e + w_s_l) / 2
within = w_bar * (m_s_l - m_s_e)
composition = total - within
NUM['decomp'] = dict(total=total, within_stayers_raw=m_s_l - m_s_e, within_weighted=within, composition=composition,
                     within_share=within / total, stayers_share_raw=(m_s_l - m_s_e) / total,
                     stay_early=m_s_e, stay_late=m_s_l, leavers_early=m_lv, entrants_late=m_en,
                     stayer_weight_early=w_s_e, stayer_weight_late=w_s_l)
pd.Series(NUM['decomp']).round(3)

In [ ]:
# --- 1.4 within-politician paired change (stayers with >= 5 speeches in each period) ----------
g = df[df.politicianId.isin(stayers)].groupby(['politicianId', 'late'])[EMI].agg(['mean', 'size']).unstack()
g = g[(g['size'][0] >= 5) & (g['size'][1] >= 5)]
chg = (g['mean'][1] - g['mean'][0]).rename('change').to_frame()
chg['leaning'] = df.groupby('politicianId')['leaning'].first().reindex(chg.index)
chg['party'] = df.groupby('politicianId')['party'].first().reindex(chg.index)
chg.to_csv(os.path.join(OUT1, 'within_politician_change.csv'))
tt = stats.ttest_1samp(chg.change, 0)
NUM['paired'] = dict(n=int(len(chg)), mean_change=float(chg.change.mean()), ci=boot_ci(chg.change.values),
                     t=float(tt.statistic), p=float(tt.pvalue), share_declining=float((chg.change < 0).mean()),
                     wilcoxon_p=float(stats.wilcoxon(chg.change).pvalue), median=float(chg.change.median()))

def summarise(by):
    s = chg.groupby(by).change.agg(['size', 'mean', 'median', lambda s: (s < 0).mean()])
    s.columns = ['n', 'mean', 'median', 'share_declining']
    s['p'] = [stats.ttest_1samp(chg.change[chg[by] == k], 0).pvalue for k in s.index]
    return s
by_lean, by_party = summarise('leaning'), summarise('party')
by_lean.to_csv(os.path.join(OUT1, 'within_change_by_leaning.csv')); by_party.to_csv(os.path.join(OUT1, 'within_change_by_party.csv'))
print(NUM['paired']); display(by_lean.round(3)); by_party.round(3)

In [ ]:
# --- 1.5 breakpoint models on the panels: random intercepts (as in the paper) and politician FE --
def prep(d):
    m = d[[EMI, 'time_quarterly', 'leaning', 'politicianId']].copy()
    q = m.groupby(['politicianId', 'time_quarterly'])[EMI].mean().rename('_q').reset_index()
    q['emi_lag1'] = q.groupby('politicianId')['_q'].shift(1)
    m = m.merge(q[['politicianId', 'time_quarterly', 'emi_lag1']], on=['politicianId', 'time_quarterly']).dropna()
    tm = m.time_quarterly.mean(); m['time_centered'] = m.time_quarterly - tm
    for k, ts in BP.items():
        bq = int(f'{ts.year}{ts.quarter}')
        m[f'slope_{k}'] = np.maximum(0, m.time_centered - (bq - tm))
        m[f'level_{k}'] = (m.time_quarterly >= bq).astype(float)
    cats = [c for c in ['middle', 'left', 'right'] if c in set(m.leaning)]
    m['leaning'] = pd.Categorical(m.leaning, categories=cats)
    return m.reset_index(drop=True), cats

def formula(cats, full, fe=False, skip_bp=()):
    t = ['time_centered'] + [f'slope_{k}' for k in BP] + [f'level_{k}' for k in BP]
    if len(cats) > 1:
        if not fe: t.append('leaning')
        t.append('time_centered:leaning')
        if full:
            t += [f'slope_{k}:leaning' for k in BP if k not in skip_bp] + [f'level_{k}:leaning' for k in BP if k not in skip_bp]
    t.append('emi_lag1')
    return f'{EMI} ~ ' + ' + '.join(t)

def tidy(res, tag, n, npol, extra=None):
    ci = res.conf_int()
    t = pd.DataFrame({'coef': res.params, 'se': res.bse, 'p': res.pvalues, 'lo': ci[0], 'hi': ci[1]})
    t['model'] = tag; t['n'] = n; t['n_pol'] = npol
    for k, v in (extra or {}).items(): t[k] = v
    return t

fits = []
for pname, ids in [('all', None), ('panelA', panelA), ('panelB', panelB)]:
    d = df if ids is None else df[df.politicianId.isin(ids)]
    m, cats = prep(d)
    # a camp with no observations before a breakpoint makes that breakpoint's leaning interactions
    # collinear with the camp main effect -> drop them (the AfD enters in 2017)
    skip = tuple(k for k, ts in BP.items()
                 if (m.groupby('leaning', observed=True).time_quarterly.min() >= int(f'{ts.year}{ts.quarter}')).any())
    NUM.setdefault('skipped_bp_interactions', {})[pname] = list(skip)
    for full in (False, True):
        r = MixedLM.from_formula(formula(cats, full, skip_bp=skip), groups=m.politicianId, data=m).fit(reml=False)
        fits.append(tidy(r, f"{pname}_{'full' if full else 'constrained'}_RI", len(m), m.politicianId.nunique(),
                         dict(group_var=r.cov_re.iloc[0, 0], scale=r.scale, llf=r.llf)))
        print(pname, 'RI', 'full' if full else 'constrained', 'N', len(m), 'pol', m.politicianId.nunique(),
              'level2017', round(r.params['level_elec2017'], 3), 'level2021', round(r.params['level_elec2021'], 3),
              'slope2021', round(r.params['slope_elec2021'], 4))
    for full in (False, True):
        f = formula(cats, full, fe=True, skip_bp=skip) + ' + C(politicianId)'
        r = smf.ols(f, data=m).fit(cov_type='cluster', cov_kwds={'groups': m.politicianId})
        t = tidy(r, f"{pname}_{'full' if full else 'constrained'}_FE", len(m), m.politicianId.nunique(), dict(r2_within=r.rsquared))
        fits.append(t[~t.index.str.startswith('C(politicianId)')])
        print(pname, 'FE', 'full' if full else 'constrained',
              'level2017', round(r.params['level_elec2017'], 3), 'level2021', round(r.params['level_elec2021'], 3),
              'slope2021', round(r.params['slope_elec2021'], 4))
models = pd.concat(fits); models.index.name = 'term'
models.to_csv(os.path.join(OUT1, 'breakpoint_models.csv'))

In [ ]:
# key coefficients side by side (level shifts and slope changes, all samples, RI vs FE)
key = ['level_elec2017', 'level_elec2021', 'level_elec2025', 'slope_elec2021', 'slope_elec2025', 'emi_lag1']
cmp = (models[models.index.isin(key)].reset_index()
       .pivot(index='term', columns='model', values='coef').reindex(key).round(3))
cmp.to_csv(os.path.join(OUT1, 'key_coefficients_comparison.csv'))
cmp

In [ ]:
# --- 1.6 figure: A quarterly EMI for all vs stayer panels, B panel B by camp, C within-politician change by party
fig, axes = plt.subplots(1, 3, figsize=(20, 5.6), gridspec_kw={'width_ratios': [1.5, 1.5, 1]})
ax = axes[0]
for name, ids, col, ls in [('All speeches', None, 'black', '-'), ('Stayers WP18–20', panelA, '#117733', '-'),
                            ('Stayers WP19–20', panelB, '#AA4499', '--')]:
    d = df if ids is None else df[df.politicianId.isin(ids)]
    q = d.groupby('qdate')[EMI].agg(['mean', 'sem'])
    ax.plot(q.index, q['mean'], color=col, ls=ls, lw=2, label=name)
    ax.fill_between(q.index, q['mean'] - 1.96 * q['sem'], q['mean'] + 1.96 * q['sem'], color=col, alpha=0.12)
for ts in BP.values(): ax.axvline(ts.tz_localize(None), color='grey', ls=':', lw=1)
ax.axhline(0, color='grey', lw=0.8); ax.set_ylabel('EMI score'); ax.set_xlabel('Date'); ax.legend(frameon=False)
ax.text(0.01, 0.97, 'A', transform=ax.transAxes, fontsize=18, fontweight='bold', va='top')

ax = axes[1]
dB = df[df.politicianId.isin(panelB)]
for lean in ['middle', 'left', 'right']:
    q = dB[dB.leaning == lean].groupby('qdate')[EMI].agg(['mean', 'sem'])
    ax.plot(q.index, q['mean'], color=COLORS[lean], lw=2, label=f'{LAB[lean]} (stayers WP19–20)')
    ax.fill_between(q.index, q['mean'] - 1.96 * q['sem'], q['mean'] + 1.96 * q['sem'], color=COLORS[lean], alpha=0.12)
for ts in BP.values(): ax.axvline(ts.tz_localize(None), color='grey', ls=':', lw=1)
ax.axhline(0, color='grey', lw=0.8); ax.set_xlabel('Date'); ax.legend(frameon=False)
ax.text(0.01, 0.97, 'B', transform=ax.transAxes, fontsize=18, fontweight='bold', va='top')

ax = axes[2]
order = [p for p in ['CDU/CSU', 'SPD', 'FDP', 'Alliance 90/The Greens', 'The Left Party', 'AFD'] if p in set(chg.party)]
data = [chg.change[chg.party == p].values for p in order]
bp = ax.boxplot(data, vert=False, patch_artist=True, showfliers=False, widths=0.6)
for patch, p in zip(bp['boxes'], order): patch.set_facecolor(PARTY_COL[p]); patch.set_alpha(0.55)
for med in bp['medians']: med.set_color('black')
ax.set_yticks(range(1, len(order) + 1))
ax.set_yticklabels([p.replace('Alliance 90/The Greens', 'Greens').replace('The Left Party', 'Left') for p in order])
ax.axvline(0, color='grey', lw=0.8); ax.set_xlabel('Within-politician change in mean EMI\n(2020–25 minus 2015–19)')
ax.text(0.01, 0.97, 'C', transform=ax.transAxes, fontsize=18, fontweight='bold', va='top')
for a in axes: a.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT1, 'stayers_figure.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(OUT1, 'stayers_figure.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 1.7 panel descriptives and save ---------------------------------------------------------
for nm, ids in [('panelA', panelA), ('panelB', panelB), ('stayers', stayers)]:
    NUM[nm] = dict(n_pol=len(ids), n_speeches=int(df.politicianId.isin(ids).sum()),
                   leanings=df[df.politicianId.isin(ids)].groupby('leaning').politicianId.nunique().to_dict())
save_numbers(NUM, OUT1)
print(json.dumps({k: NUM[k] for k in ['decomp', 'paired', 'panelA', 'panelB']}, indent=1, default=float))

**Reading the output.** The within-stayer change carries the bulk of the early-to-late decline (see `decomp`),
the paired within-politician change is negative for the large majority of stayers, and the 2017 and 2021 level shifts
keep their sign and approximate size on panel A (which cannot contain any AfD member) and under politician fixed effects.
These are the numbers reported in the Supplement (composition analysis).

## 2. Pre-2015 extension: is 2017 special, or is every election a break?

Is the 2017 level shift specific to the AfD's entry or an election-cycle
regularity, given that EMI drops at every election? We score **all Bundestag plenary speeches from
2000 to mid-2025** with the paper's parliament model (`code/03_scoring/0304_score_emi.py`, same dictionary, same tokenizer, same
length-bin adjustment) and standardise **once over the whole 2000–2025 set**, so the pre- and post-2015 values are on
one scale. Then we (i) check that the 2015–2025 subset reproduces the published series, (ii) estimate level shifts at
all seven federal elections (2002, 2005, 2009, 2013, 2017, 2021, 2025) with the paper's constrained breakpoint model,
(iii) compare against placebo breakpoints at random dates and a data-driven single-break search.

Inputs: `data/revision/bundestag_2000_2025_input.csv` (from `code/01_data/parliament/0115_build_input_2000_2025.py`: the paper's own 59,170 rows for
2015–2025, untouched, plus Open Discourse faction speeches 2000–2014 extracted by `code/01_data/parliament/0114_extract_bundestag_2000.py` and
cleaned with the paper's preprocessing) and `data/revision/bundestag_2000_2025_scored.csv` (raw cosine scores from `code/03_scoring/0304_score_emi.py`).

**Model note.** The archive `parliament_model/` is the 300-dimensional SBERT-wrapped word2vec model. On the stored
2015–2025 scores it reproduces the ranking (r = 0.94 evidence, 0.95 intuition, 0.97 EMI on 5,000 speeches) but with a
uniform offset, so the stored scores came from a different training run of the same pipeline (the Supplement describes
a 100-dimensional model). All numbers in this section are therefore from one consistent re-scoring; the 2015–2025
subset is compared with the published series in 2.1.

In [ ]:
OUT2 = os.path.join(RES, '02_pre2015'); os.makedirs(OUT2, exist_ok=True)
NUM2 = {}
ELECTIONS = {'elec2002': '2002-09-22', 'elec2005': '2005-09-18', 'elec2009': '2009-09-27', 'elec2013': '2013-09-22',
             'elec2017': '2017-09-24', 'elec2021': '2021-09-26', 'elec2025': '2025-02-23'}
# breakpoints follow the paper's convention (1 July of the election year; 1 January 2025 for the February 2025 election)
BP7 = {k: pd.Timestamp(f'{v[:4]}-07-01' if k != 'elec2025' else '2025-01-01', tz='UTC') for k, v in ELECTIONS.items()}

# input built by build_input_2000_2025.py: the paper's own 59,170 rows for 2015-2025 (source = paper_corpus, with the
# published scores as *_paper) + Open Discourse faction speeches 2000-2014 cleaned the same way; scored by score_emi.py
sc = pd.read_csv(os.path.join(HERE, 'data', 'revision', 'bundestag_2000_2025_scored.csv'), dtype={'id': str, 'politicianId': str})
sc['date'] = pd.to_datetime(sc['date'], utc=True)
full = sc[(sc.length > 10)].copy()
full['time_quarterly'] = (full.date.dt.year * 10 + full.date.dt.quarter).astype(int)
full['qdate'] = full.date.dt.tz_localize(None).dt.to_period('Q').dt.to_timestamp()
full['year'] = full.date.dt.year
NUM2['n_scored'] = int(len(sc)); NUM2['n_kept'] = int(len(full)); NUM2['n_speakers'] = int(full.politicianId.nunique())
NUM2['n_by_source'] = full.groupby('source').size().to_dict()
print('speeches', len(full), full.date.min().date(), '-', full.date.max().date(), '| speakers', full.politicianId.nunique())
print(full.groupby('source').size().to_dict()); print(full.groupby('term').size().to_dict())

# --- length-bin adjustment and z-standardisation ONCE over 2000-2025 (paper's procedure) ----------
def adjust_and_z(d, tag):
    d = d.copy(); bins = range(0, int(d.length.max()) + 10, 10)
    lb = pd.cut(d.length, bins=bins)
    for c in ['evidence', 'intuition']:
        d[f'{c}_adj'] = d[f'{c}_score'] - d.groupby(lb, observed=True)[f'{c}_score'].transform('mean')
        d[f'{c}_z_{tag}'] = (d[f'{c}_adj'] - d[f'{c}_adj'].mean()) / d[f'{c}_adj'].std()
    d[f'emi_{tag}'] = d[f'evidence_z_{tag}'] - d[f'intuition_z_{tag}']
    return d.drop(columns=['evidence_adj', 'intuition_adj'])
full = adjust_and_z(full, 'full')                                   # reference set: 2000-2025
sub = adjust_and_z(full[full.date >= '2015-01-01'], 'p15')          # reference set: 2015-2025 (paper window)
full = full.merge(sub[['id', 'emi_p15']], on='id', how='left')
full.drop(columns=['text_clean']).to_csv(os.path.join(OUT2, 'bundestag_2000_2025_emi_harmonised.csv'), index=False)
full[['emi_full', 'evidence_score', 'intuition_score']].describe().round(3)

### 2.1 Does the re-scored 2015–2025 subset reproduce the published series?

In [ ]:
# same rows as the paper for 2015-2025: compare speech by speech and quarter by quarter
pc = full[full.source == 'paper_corpus']
NUM2['replication'] = dict(
    n_paper_rows=int(len(pc)),
    r_speech_evidence=float(pc.evidence_score.corr(pc.evidence_score_paper)),
    r_speech_intuition=float(pc.intuition_score.corr(pc.intuition_score_paper)),
    r_speech_emi=float(pc.emi_p15.corr(pc.emi_paper)))
q = pc.groupby('qdate').agg(emi_paper=('emi_paper', 'mean'), emi_rescored_p15=('emi_p15', 'mean'), emi_rescored_fullz=('emi_full', 'mean'))
NUM2['replication']['r_quarterly'] = float(q.emi_paper.corr(q.emi_rescored_p15)); NUM2['replication']['n_quarters'] = int(len(q))
party_old = pc.groupby('party')['emi_paper'].mean().sort_values(ascending=False)
party_new = pc.groupby('party')['emi_p15'].mean().sort_values(ascending=False)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in NUM2['replication'].items()})
print('party order paper   :', list(party_old.index)); print('party order rescored:', list(party_new.index))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(q.index, q.emi_paper, 'k-', lw=2, label='published scores (parliament_nhb.csv)')
ax.plot(q.index, q.emi_rescored_p15, color='#D55E00', lw=2, ls='--', label='re-scored, same speeches, z over 2015–2025')
ax.axhline(0, color='grey', lw=0.8); ax.set_ylabel('quarterly mean EMI'); ax.legend(frameon=False); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT2, 'replication_2015_2025.png'), dpi=150); plt.show()

### 2.2 Quarterly EMI 2000–2025 and simple pre/post comparisons at every election

In [ ]:
# quarterly means overall and by camp, standardised over 2000-2025
qa = full.groupby('qdate')['emi_full'].agg(['mean', 'sem', 'size'])
qc = full.groupby(['leaning', 'qdate'])['emi_full'].agg(['mean', 'sem', 'size']).reset_index()
qa.to_csv(os.path.join(OUT2, 'quarterly_all.csv')); qc.to_csv(os.path.join(OUT2, 'quarterly_by_leaning.csv'), index=False)

# pre/post comparison: 6 full quarters before the election quarter vs 6 full quarters after the constitution quarter,
# skipping the two transition quarters (election quarter and the following one), as in the sparse-quarter check
rows = []
for k, dstr in ELECTIONS.items():
    e = pd.Timestamp(dstr, tz='UTC'); eq = e.to_period('Q')
    pre = full[(full.date.dt.to_period('Q') < eq) & (full.date.dt.to_period('Q') >= eq - 6)]
    post = full[(full.date.dt.to_period('Q') > eq + 1) & (full.date.dt.to_period('Q') <= eq + 7)]
    t, p = stats.ttest_ind(pre.emi_full, post.emi_full, equal_var=False)
    rows.append(dict(election=k, date=dstr, n_pre=len(pre), n_post=len(post), mean_pre=pre.emi_full.mean(),
                     mean_post=post.emi_full.mean(), diff=post.emi_full.mean() - pre.emi_full.mean(), t=t, p=p,
                     ci=boot_ci(post.emi_full.values) if len(post) else None))
prepost = pd.DataFrame(rows); prepost.to_csv(os.path.join(OUT2, 'prepost_by_election.csv'), index=False)
NUM2['prepost'] = prepost.set_index('election')[['n_pre', 'n_post', 'mean_pre', 'mean_post', 'diff', 'p']].to_dict('index')
prepost[['election', 'n_pre', 'n_post', 'mean_pre', 'mean_post', 'diff', 't', 'p']].round(3)

In [ ]:
# rolling-window series (centred 180-day window, 14-day steps). A point is drawn only when the window holds at
# least MIN_N speeches of that camp, so periods without observations (e.g. no right camp before 2017) stay blank.
WINDOW, STEP, MIN_N = 180, 14, 30
def rolling_series(d, y='emi_full'):
    d = d.sort_values('date'); dates = d.date.dt.tz_localize(None).values.astype('datetime64[ns]'); vals = d[y].values
    grid = pd.date_range(full.date.min().tz_localize(None), full.date.max().tz_localize(None), freq=f'{STEP}D')
    half = np.timedelta64(WINDOW // 2, 'D'); out = []
    for g in grid:
        lo, hi = np.searchsorted(dates, g.to_datetime64() - half), np.searchsorted(dates, g.to_datetime64() + half)
        w = vals[lo:hi]
        out.append((g, w.mean(), w.std(ddof=1) / np.sqrt(len(w)), len(w)) if len(w) >= MIN_N else (g, np.nan, np.nan, len(w)))
    return pd.DataFrame(out, columns=['date', 'mean', 'sem', 'n']).set_index('date')

roll_all = rolling_series(full)
roll_camp = {lean: rolling_series(full[full.leaning == lean]) for lean in ['middle', 'left', 'right']}
roll_all.to_csv(os.path.join(OUT2, 'rolling_all.csv'))
pd.concat(roll_camp, names=['leaning']).to_csv(os.path.join(OUT2, 'rolling_by_leaning.csv'))

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
ax = axes[0]
ax.plot(roll_all.index, roll_all['mean'], color='black', lw=2)
ax.fill_between(roll_all.index, roll_all['mean'] - 1.96 * roll_all['sem'], roll_all['mean'] + 1.96 * roll_all['sem'], color='black', alpha=0.12)
for k, ts in BP7.items():
    ax.axvline(ts.tz_localize(None), color='#D55E00' if k == 'elec2017' else 'grey', ls=':', lw=1.4)
ax.axhline(0, color='grey', lw=0.8); ax.set_ylabel('EMI (z over 2000–2025)')
ax.set_title(f'All speeches, centred {WINDOW}-day rolling mean with 95% CI')
ax.text(0.01, 0.95, 'A', transform=ax.transAxes, fontsize=16, fontweight='bold', va='top')
ax = axes[1]
for lean in ['middle', 'left', 'right']:
    r = roll_camp[lean]                                   # NaN rows break the line: no observations, no line
    ax.plot(r.index, r['mean'], color=COLORS[lean], lw=2, label=LAB[lean])
    ax.fill_between(r.index, r['mean'] - 1.96 * r['sem'], r['mean'] + 1.96 * r['sem'], color=COLORS[lean], alpha=0.12)
for k, ts in BP7.items():
    ax.axvline(ts.tz_localize(None), color='#D55E00' if k == 'elec2017' else 'grey', ls=':', lw=1.4)
ax.axhline(0, color='grey', lw=0.8); ax.set_ylabel('EMI (z over 2000–2025)'); ax.set_xlabel('Date'); ax.legend(frameon=False)
ax.set_title(f'By camp (window drawn only with at least {MIN_N} speeches)')
ax.text(0.01, 0.95, 'B', transform=ax.transAxes, fontsize=16, fontweight='bold', va='top')
for a in axes: a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT2, 'emi_2000_2025.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(OUT2, 'emi_2000_2025.png'), dpi=150, bbox_inches='tight'); plt.show()

### 2.3 Constrained breakpoint model with all seven elections (paper's Eq. 2, random speaker intercepts)

In [ ]:
def prep7(d, bps, y='emi_full'):
    m = d[[y, 'time_quarterly', 'leaning', 'politicianId']].rename(columns={y: 'y'}).copy()
    qm = m.groupby(['politicianId', 'time_quarterly'])['y'].mean().rename('_q').reset_index()
    qm['y_lag1'] = qm.groupby('politicianId')['_q'].shift(1)
    m = m.merge(qm[['politicianId', 'time_quarterly', 'y_lag1']], on=['politicianId', 'time_quarterly']).dropna()
    tm = m.time_quarterly.mean(); m['time_centered'] = m.time_quarterly - tm
    for k, ts in bps.items():
        bq = int(f'{ts.year}{ts.quarter}')
        m[f'slope_{k}'] = np.maximum(0, m.time_centered - (bq - tm)); m[f'level_{k}'] = (m.time_quarterly >= bq).astype(float)
    m['leaning'] = pd.Categorical(m.leaning, categories=['middle', 'left', 'right'])
    return m.reset_index(drop=True)

m7 = prep7(full, BP7)
f7 = 'y ~ time_centered + leaning + time_centered:leaning + ' + ' + '.join([f'slope_{k} + level_{k}' for k in BP7]) + ' + y_lag1'
r7 = MixedLM.from_formula(f7, groups=m7.politicianId, data=m7).fit(reml=False)
ci = r7.conf_int()
tab7 = pd.DataFrame({'coef': r7.params, 'se': r7.bse, 'p': r7.pvalues, 'lo': ci[0], 'hi': ci[1]})
tab7.to_csv(os.path.join(OUT2, 'breakpoint7_constrained_RI.csv'))
NUM2['breakpoint7'] = {k: dict(level=float(r7.params[f'level_{k}']), level_p=float(r7.pvalues[f'level_{k}']),
                               slope=float(r7.params[f'slope_{k}']), slope_p=float(r7.pvalues[f'slope_{k}'])) for k in BP7}
print('N', len(m7), 'speakers', m7.politicianId.nunique())
tab7.loc[[f'level_{k}' for k in BP7] + [f'slope_{k}' for k in BP7] + ['y_lag1']].round(4)

In [ ]:
# forest plot of the level shifts at the seven elections
lv = tab7.loc[[f'level_{k}' for k in BP7]]
fig, ax = plt.subplots(figsize=(7, 4))
yy = np.arange(len(lv))[::-1]
ax.errorbar(lv.coef, yy, xerr=[lv.coef - lv.lo, lv.hi - lv.coef], fmt='o', color='black', capsize=3)
ax.scatter(lv.coef[lv.index == 'level_elec2017'], yy[lv.index == 'level_elec2017'], color='#D55E00', zorder=5, s=60)
ax.set_yticks(yy); ax.set_yticklabels([k.replace('elec', '') for k in BP7]); ax.axvline(0, color='grey', lw=0.8)
ax.set_xlabel('Level shift in EMI at the election breakpoint (95% CI)'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT2, 'level_shifts_forest.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(OUT2, 'level_shifts_forest.png'), dpi=150, bbox_inches='tight'); plt.show()

### 2.4 Placebo breakpoints and data-driven single-break search

Both run on the speaker × quarter panel (mean EMI per speaker and quarter, lagged value as control, speaker fixed
effects via demeaning) so that hundreds of fits are cheap. The placebo distribution is the set of level-shift
estimates for single breakpoints placed at random quarters at least two quarters away from any election quarter.

In [ ]:
panel = (full.groupby(['politicianId', 'time_quarterly'])
             .agg(y=('emi_full', 'mean'), n=('emi_full', 'size'), leaning=('leaning', 'first')).reset_index())
panel['y_lag1'] = panel.groupby('politicianId')['y'].shift(1); panel = panel.dropna().reset_index(drop=True)
panel['t'] = panel.time_quarterly.rank(method='dense')          # consecutive quarter index
quarters = np.sort(panel.time_quarterly.unique())
elec_q = {int(f'{ts.year}{ts.quarter}') for ts in BP7.values()}
def qidx(q): return int(np.searchsorted(quarters, q))

def single_break_fit(bq):
    d = panel.copy(); d['level'] = (d.time_quarterly >= bq).astype(float)
    d['slope'] = np.maximum(0, d.t - qidx(bq))
    for c in ['y', 'y_lag1', 'level', 'slope', 't']:            # speaker fixed effects by within-transformation
        d[c + '_w'] = d[c] - d.groupby('politicianId')[c].transform('mean')
    r = smf.ols('y_w ~ t_w + level_w + slope_w + y_lag1_w - 1', data=d).fit(cov_type='cluster', cov_kwds={'groups': d.politicianId})
    return float(r.params['level_w']), float(r.bse['level_w']), float(r.ssr)

# real elections, one at a time (single-break specification)
real = {k: single_break_fit(int(f'{ts.year}{ts.quarter}')) for k, ts in BP7.items()}
# placebos
rng = np.random.default_rng(0)
cands = [q for q in quarters[8:-8] if all(abs(qidx(q) - qidx(e)) > 2 for e in elec_q)]
plac = pd.DataFrame([(q, *single_break_fit(q)) for q in rng.choice(cands, size=min(150, len(cands)), replace=False)],
                    columns=['quarter', 'level', 'se', 'ssr'])
plac.to_csv(os.path.join(OUT2, 'placebo_single_break.csv'), index=False)
NUM2['placebo'] = dict(n=int(len(plac)), mean=float(plac.level.mean()), sd=float(plac.level.std()),
                       q05=float(plac.level.quantile(.05)), q95=float(plac.level.quantile(.95)),
                       real={k: dict(level=v[0], se=v[1], rank_pct=float((plac.level < v[0]).mean())) for k, v in real.items()})
print(pd.DataFrame({k: dict(level=v[0], se=v[1], share_placebos_more_negative=float((plac.level < v[0]).mean())) for k, v in real.items()}).T.round(3))

# data-driven single break: the quarter whose single-break model has the smallest residual sum of squares
grid = pd.DataFrame([(q, *single_break_fit(q)) for q in quarters[8:-8]], columns=['quarter', 'level', 'se', 'ssr'])
grid.to_csv(os.path.join(OUT2, 'single_break_grid.csv'), index=False)
best = grid.loc[grid.ssr.idxmin()]; NUM2['best_single_break'] = dict(quarter=int(best.quarter), level=float(best.level))
print('best single break:', int(best.quarter), 'level', round(best.level, 3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(plac.level, bins=25, color='grey', alpha=0.7)
for k, v in real.items(): axes[0].axvline(v[0], color='#D55E00' if k == 'elec2017' else 'black', lw=1.5)
axes[0].set_xlabel('level shift, placebo single breaks (grey) vs elections (lines; 2017 orange)')
axes[1].plot(pd.PeriodIndex([f'{q//10}Q{q%10}' for q in grid.quarter], freq='Q').to_timestamp(), grid.ssr, color='black')
for ts in BP7.values(): axes[1].axvline(ts.tz_localize(None), color='grey', ls=':')
axes[1].set_ylabel('RSS of single-break model'); axes[1].set_xlabel('candidate break quarter')
for a in axes: a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT2, 'placebo_and_break_search.png'), dpi=150, bbox_inches='tight'); plt.show()
save_numbers(NUM2, OUT2)

**Reading the output.** Section 2.1 shows how closely the re-scored 2015–2025 subset tracks the published series.
Section 2.2 and the forest plot in 2.3 answer this question directly: if the 2002–2013 transitions show level
shifts of the same sign and size as 2017, the parliamentary pattern is an election-cycle regularity and the paper says
so; if 2017 stands out against both the earlier elections and the placebo distribution in 2.4, that is the evidence
that 2017 is unusual, still without identifying the AfD as its cause.

## 2b. 2017 sparse-quarter check
*Planned:* re-estimate Eq. 2/3 with 2017Q3 and 2017Q4 removed, and with the breakpoint at the constitution of the
19th Bundestag (24 Oct 2017); report both next to the main specification.

## 3. Grouped lexicon ablation
*Planned:* dictionary groups Intuition-core / Honesty / Deception-accusation / Scepticism / Evidence-procedural;
leave-one-group-out and core-only EMI; re-fit all main models; forest plot of trend and breakpoint coefficients.

## 4. Self- vs other-attribution
*Planned:* stratified 5–10k sentences, LLM coding into own justification / criticism of others' reasoning /
credibility accusation / neutral; validation on ~500 double-coded sentences (Krippendorff's α); trend with and without
accusation sentences.

## 5. Discriminant validity
*Planned:* moralisation, emotionality, populism, negativity on the same documents; correlation matrix; models with
controls and with residualised EMI.

## 6. Hyperparameter multiverse
*Planned:* word2vec 50/100/200/300 dims × window 5/10 × 3 seeds; correlations, main coefficients, validation AUC.

## 7. Document-level validation: is EMI equally valid across parties and periods?

Is the EMI score equally reliable across parties and time periods? The document-level
validation survey (Supplementary Note 2.11) rated 1,300 parliamentary chunks of 10–50 tokens that were sampled in
score bins **across all decades of the historical corpus (1860s–2020s)**, so the same survey allows the validity check
by period and, through the speech metadata, by faction. Labels are rebuilt here from the raw Potato exports in
`data/validation/annotations.zip` with the 32 raters who failed the attention check excluded (the local
`survey-data-analysis.ipynb` had that exclusion commented out). Chunk scores are the chunk-level `evidence_z`,
`intuition_z` and `emi` from the original scoring run (`validation_results.csv`); within a group AUC is rank-based, so
the standardisation reference does not matter. Faction codes are resolved to party names through the full parliament
table (`code/05_validation/document/0516_lookup_validation_parties.py`).

In [ ]:
from sklearn.metrics import roc_auc_score
OUT7 = os.path.join(RES, '07_validation_auc'); os.makedirs(OUT7, exist_ok=True)
NUM7 = {}
VD = os.path.join(DATA, 'validation', 'unzipped')
if not os.path.exists(VD):
    import zipfile; zipfile.ZipFile(os.path.join(DATA, 'validation', 'annotations.zip')).extractall(VD)

# --- rebuild instance labels from the raw annotation exports ------------------------------------
import glob
ann = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(VD, 'annotated_instances*.csv')))], ignore_index=True)
ok = {'Stimme voll und ganz zu', 'Stimme zu', 'Neutral'}
failed = ann[((ann.instance_id == '1_testing') & ~ann['Rating:::Intuition basiert'].isin(ok)) |
             ((ann.instance_id == '0_testing') & ~ann['Rating:::Evidenz basiert'].isin(ok))].user.unique()
NUM7['raters_total'] = int(ann.user.nunique()); NUM7['raters_failed'] = int(len(failed))
ann = ann[~ann.user.isin(failed) & ~ann.instance_id.isin(['0_testing', '1_testing', 'EMI-consent.html'])].copy()
rmap = {'Stimme überhaupt nicht zu': -2, 'Stimme nicht zu': -1, 'Neutral': 0, 'Stimme zu': 1, 'Stimme voll und ganz zu': 2}
ann['ev'] = ann['Rating:::Evidenz basiert'].map(rmap); ann['iu'] = ann['Rating:::Intuition basiert'].map(rmap)
lab = ann.groupby('instance_id').agg(n_raters=('user', 'size'), ev_mean=('ev', 'mean'), iu_mean=('iu', 'mean'),
                                     ev_share=('ev', lambda s: (s >= 1).mean()), iu_share=('iu', lambda s: (s >= 1).mean())).reset_index()
lab['id'] = lab.instance_id.astype(str)
lab['ev_label'] = (lab.ev_share > 0.5).astype(int)            # majority of raters endorse 'evidence-based'
lab['iu_label'] = (lab.iu_share > 0.5).astype(int)            # majority endorse 'intuition-based'
lab['emi_label'] = ((lab.ev_mean - lab.iu_mean) > 0).astype(int)   # survey EMI > 0

# --- chunk scores and speech metadata from the original run ------------------------------------
vr = pd.read_csv(os.path.join(VD, 'validation_results.csv'), dtype={'id': str})
val = lab.merge(vr[['id', 'decade', 'year', 'date', 'electoralTerm', 'factionId', 'politicianId', 'chunk_length',
                    'evidence_score', 'intuition_score', 'evidence_z', 'intuition_z', 'emi']], on='id', how='left')
pp = pd.read_csv(os.path.join(HERE, 'data', 'revision', 'validation_chunk_party.csv'), dtype={'id': str})
val = val.merge(pp[['id', 'party', 'party_orientation', 'od_date']], on='id', how='left')
# chunks from WP 18-19 (official-XML era) and pre-1949 sittings carry no Open Discourse id in the full table:
# map their Open Discourse factionId code to a party name, using the resolved rows plus the known OD codes
fmap = val.dropna(subset=['party']).groupby('factionId').party.agg(lambda s: s.mode()[0]).to_dict()
fmap.update({k: v for k, v in {0: 'AfD', 3: 'BÜNDNIS 90/DIE GRÜNEN', 4: 'CDU/CSU', 6: 'DIE LINKE', 13: 'FDP', 23: 'SPD'}.items() if k not in fmap})
val['party_source'] = np.where(val.party.notna(), 'full_table', np.where(val.factionId.map(fmap).notna(), 'factionId_code', 'none'))
val['party'] = val.party.fillna(val.factionId.map(fmap))
print('faction code map used:', {int(k): v for k, v in sorted(fmap.items()) if k in (0, 3, 4, 6, 13, 23)})
val['year'] = val.year.fillna(pd.to_datetime(val.od_date, errors='coerce').dt.year)
val['era'] = pd.cut(val.year, bins=[1866, 1918, 1932, 1945, 1990, 2014, 2026],
                    labels=['Kaiserreich 1867–1918', 'Weimar 1919–1932', 'NS 1933–1945', 'Bonn 1949–1990', 'Berlin 1991–2014', '2015–2025'])
NUM7['n_instances'] = int(len(val)); NUM7['n_with_party'] = int(val.party.notna().sum())
print('instances', len(val), '| with party', val.party.notna().sum(), '| raters per instance', val.n_raters.value_counts().sort_index().to_dict())
print('label base rates: evidence %.3f intuition %.3f emi %.3f' % (val.ev_label.mean(), val.iu_label.mean(), val.emi_label.mean()))
val.to_csv(os.path.join(OUT7, 'validation_instances.csv'), index=False)

In [ ]:
# --- AUC with bootstrap CI, overall and by group -------------------------------------------------
PAIRS = [('evidence', 'ev_label', 'evidence_z'), ('intuition', 'iu_label', 'intuition_z'), ('EMI', 'emi_label', 'emi')]

def auc_ci(y, s, n_boot=1000, seed=0):
    y = np.asarray(y); s = np.asarray(s); m = ~np.isnan(s); y, s = y[m], s[m]
    if len(y) < 10 or y.min() == y.max():
        return np.nan, np.nan, np.nan, len(y)
    a = roc_auc_score(y, s); r = np.random.default_rng(seed); bs = []
    for _ in range(n_boot):
        i = r.integers(0, len(y), len(y))
        if y[i].min() != y[i].max(): bs.append(roc_auc_score(y[i], s[i]))
    return a, float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)), len(y)

def auc_table(df, group, min_n=20):
    rows = []
    for g, d in ([('all', df)] if group is None else df.groupby(group, observed=True)):
        if len(d) < min_n: continue
        for name, yl, sc in PAIRS:
            a, lo, hi, n = auc_ci(d[yl], d[sc])
            rows.append(dict(group=str(g), construct=name, n=n, pos_rate=float(d[yl].mean()), auc=a, lo=lo, hi=hi))
    return pd.DataFrame(rows)

overall = auc_table(val, None); print(overall.round(3).to_string(index=False))
NUM7['overall'] = overall.set_index('construct')[['auc', 'lo', 'hi', 'n']].to_dict('index')
by_era = auc_table(val, 'era'); by_decade = auc_table(val, 'decade')
val['party_merged'] = val.party.replace({'DIE LINKE': 'DIE LINKE / PDS', 'PDS': 'DIE LINKE / PDS'})   # PDS = predecessor of the Left Party
by_party = auc_table(val, 'party_merged')
# the paper's three camps within the 2015-2025 window (small n: report, do not over-interpret)
camp = {'AfD': 'right', 'DIE LINKE': 'left', 'BSW': 'left', 'CDU/CSU': 'middle', 'SPD': 'middle', 'FDP': 'middle', 'BÜNDNIS 90/DIE GRÜNEN': 'middle'}
v15 = val[val.year >= 2015].assign(camp=lambda d: d.party.map(camp))
by_camp15 = auc_table(v15, 'camp', min_n=10)
for nm, t in [('by_era', by_era), ('by_decade', by_decade), ('by_party', by_party), ('by_camp_2015_2025', by_camp15)]:
    t.to_csv(os.path.join(OUT7, f'auc_{nm}.csv'), index=False)
    NUM7[nm] = {f'{r.group}|{r.construct}': dict(auc=r.auc, lo=r.lo, hi=r.hi, n=int(r.n)) for r in t.itertuples()}
print(); print(by_era.pivot(index='group', columns='construct', values='auc').round(3))
print(); print(by_party.pivot(index='group', columns='construct', values='auc').round(3).join(by_party[by_party.construct == 'EMI'].set_index('group').n))
print(); print(by_camp15.round(3).to_string(index=False))

In [ ]:
# --- forest plots of the EMI AUC by era, party and decade ----------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6), gridspec_kw={'width_ratios': [1, 1, 1.2]})
for ax, (title, t, order) in zip(axes, [
        ('By era', by_era, list(val.era.cat.categories)),
        ('By party (n ≥ 20 chunks)', by_party, by_party[by_party.construct == 'EMI'].sort_values('n', ascending=False).group.tolist()),
        ('By decade', by_decade, sorted(by_decade.group.unique(), key=float))]):
    t = t[t.construct == 'EMI'].set_index('group').reindex([o for o in order if o in set(t.group)])
    yy = np.arange(len(t))[::-1]
    ax.errorbar(t.auc, yy, xerr=[t.auc - t.lo, t.hi - t.auc], fmt='o', color='black', capsize=3)
    ax.axvline(0.5, color='grey', lw=0.8, ls='--'); ax.axvline(NUM7['overall']['EMI']['auc'], color='#D55E00', lw=1.2, ls=':')
    ax.set_yticks(yy); ax.set_yticklabels([f'{g} (n={int(n)})' for g, n in zip(t.index, t.n)])
    ax.set_xlim(0.3, 1.0); ax.set_xlabel('AUC of EMI against the human EMI label (95% bootstrap CI)'); ax.set_title(title); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT7, 'auc_by_group.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(OUT7, 'auc_by_group.png'), dpi=150, bbox_inches='tight'); plt.show()
save_numbers(NUM7, OUT7)

### 7.1 Correlations between computed scores and survey items by era, decade and party

Same chunks, continuous version of the check: Spearman ρ between the computed score and the mean human rating
(evidence score vs mean evidence rating, intuition score vs mean intuition rating, EMI vs survey EMI = mean evidence
minus mean intuition rating), pooled and by group, with 95% bootstrap CIs. Implemented in `code/05_validation/document/0514_validation_correlations.py`
so the Supplement draws on one file.

In [ ]:
import runpy
runpy.run_path(os.path.join(HERE, 'code/05_validation/document/0514_validation_correlations.py'))
corr = {nm: pd.read_csv(os.path.join(OUT7, f'corr_{nm}.csv')) for nm in ['pooled', 'by_era', 'by_decade', 'by_party']}
NUM7['correlations'] = {nm: {f'{r.group}|{r.construct}': dict(rho=r.rho, lo=r.lo, hi=r.hi, n=int(r.n)) for r in t.itertuples()} for nm, t in corr.items()}
save_numbers(NUM7, OUT7)
from IPython.display import Image, display
display(corr['by_era'].pivot(index='group', columns='construct', values='rho').round(2))
display(corr['by_party'].pivot(index='group', columns='construct', values='rho').round(2))
display(Image(os.path.join(OUT7, 'corr_by_group.png')))

**Reading the output.** The orange dotted line is the pooled EMI AUC; a group whose interval covers it shows no
evidence of differential validity. Group sizes are 50–72 chunks per decade and 20–190 per party, so intervals are wide
by construction; the useful statements are whether any party or era falls to chance (AUC near 0.5) and whether the
2015–2025 window, which the paper's claims rest on, performs at least as well as the earlier eras.

## 8. Topic controls without dropping the noise cluster, on identical samples

The original Supplement reports the BERTopic noise cluster (topic −1; 42.9% of speeches, 50.8% of the
Twitter topic subsample) as excluded from the topic-controlled regressions; here we report exclusion rates by party and
time and compare models with and without topic controls on the same retained sample. Here the noise cluster is kept
as its own category (a dummy like any other topic; the largest *assigned* topic is the reference), and four models are
fitted on one identical sample: baseline, baseline + topic FE (noise kept), and the same pair on the sample without
noise documents. Parliament uses the paper's constrained (Eq. 2) and full (Eq. 3) breakpoint models; Twitter is run by
`code/05_validation/topics/0542_topic_twitter_same_sample.py` (3.7 million tweets, too slow for the notebook) and its results are read in below.

In [ ]:
OUT8 = os.path.join(RES, '08_topic_controls'); os.makedirs(OUT8, exist_ok=True)
NUM8 = {}
# --- 8.1 noise-cluster shares by party and period (Parliament; Twitter from the background job) -------
tp = pd.read_csv(os.path.join(DATA, 'parliament_topics_nhb.csv'), usecols=['id', 'topic'], dtype={'id': str})
par_ids = pd.read_csv(os.path.join(DATA, 'parliament_nhb.csv'), usecols=['id', 'date', 'party', 'leaning', 'politicianId', EMI], dtype={'id': str})
par_ids['date'] = pd.to_datetime(par_ids['date'], utc=True)
pt = par_ids.merge(tp, on='id', how='inner').dropna(subset=['politicianId', 'party', 'leaning', EMI])
pt['politicianId'] = pt.politicianId.astype(int)
pt['time_quarterly'] = (pt.date.dt.year * 10 + pt.date.dt.quarter).astype(int)
pt['noise'] = (pt.topic == -1); pt['year'] = pt.date.dt.year
pt['period'] = pd.cut(pt.year, [2014, 2017, 2021, 2025], labels=['2015–2017', '2018–2021', '2022–2025'])
excl_party = pt.groupby('party').noise.agg(['mean', 'size']).rename(columns={'mean': 'noise_share', 'size': 'n'})
excl_period = pt.groupby('period', observed=True).noise.agg(['mean', 'size']).rename(columns={'mean': 'noise_share', 'size': 'n'})
excl_pp = pt.pivot_table(index='party', columns='period', values='noise', aggfunc='mean', observed=True)
excl_party.to_csv(os.path.join(OUT8, 'parliament_noise_share_by_party.csv')); excl_period.to_csv(os.path.join(OUT8, 'parliament_noise_share_by_period.csv'))
excl_pp.to_csv(os.path.join(OUT8, 'parliament_noise_share_party_by_period.csv'))
NUM8['parliament'] = dict(n_with_topics=int(len(pt)), n_baseline=int(len(par_ids)), noise_share=float(pt.noise.mean()),
                          by_party=excl_party.noise_share.round(3).to_dict(), by_period=excl_period.noise_share.round(3).to_dict(),
                          emi_noise=float(pt.loc[pt.noise, EMI].mean()), emi_assigned=float(pt.loc[~pt.noise, EMI].mean()))
print('speeches with topics', len(pt), 'of', len(par_ids), '| noise share', round(pt.noise.mean(), 3),
      '| mean EMI noise vs assigned', round(NUM8['parliament']['emi_noise'], 3), round(NUM8['parliament']['emi_assigned'], 3))
tw_path = os.path.join(HERE, 'data', 'revision', 'twitter_noise_share_year_party.csv')
if os.path.exists(tw_path):
    tw = pd.read_csv(tw_path, index_col=[0, 1]).iloc[:, 0].unstack()
    tw.to_csv(os.path.join(OUT8, 'twitter_noise_share_year_by_party.csv')); print('Twitter noise share by year x party saved')
display(excl_pp.round(3)); excl_period.round(3)

In [ ]:
# --- 8.2 Parliament: four fits on one identical sample (run by topic_parliament_same_sample.py; Twitter by
#         topic_twitter_same_sample.py). Both write to results/revision/08_topic_controls/.
import runpy
if not os.path.exists(os.path.join(OUT8, 'parliament_same_sample_models.csv')):
    runpy.run_path(os.path.join(HERE, 'code/05_validation/topics/0541_topic_parliament_same_sample.py'))
res8 = pd.read_csv(os.path.join(OUT8, 'parliament_same_sample_models.csv'), index_col=0)
key = ['level_elec2017', 'level_elec2021', 'level_elec2025', 'slope_elec2021', 'slope_elec2025', 'leaning[T.right]', 'leaning[T.left]', 'time_centered', 'emi_lag1']
cmp8 = res8[res8.index.isin(key)].reset_index().pivot(index='term', columns='model', values='coef').reindex(key)
NUM8['parliament_models'] = {m: dict(n=int(g.n.iloc[0]), group_var=float(g.group_var.iloc[0]), converged=bool(g.converged.iloc[0]),
                                     level2017=float(g.loc['level_elec2017', 'coef']), level2021=float(g.loc['level_elec2021', 'coef']),
                                     level2025=float(g.loc['level_elec2025', 'coef']), slope2021=float(g.loc['slope_elec2021', 'coef']))
                             for m, g in res8.groupby('model')}
display(res8.groupby('model')[['n', 'n_pol', 'group_var', 'converged']].first())
cmp8.round(4)

In [ ]:
# --- 8.3 figure: level shifts under the four specifications (constrained model) + noise shares ------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={'width_ratios': [1.3, 1]})
ax = axes[0]
specs = ['constrained|all|baseline', 'constrained|all|topicFE_noise_kept', 'constrained|no_noise|baseline', 'constrained|no_noise|topicFE']
labels = ['baseline, all speeches', '+ topic FE, noise kept', 'baseline, noise dropped', '+ topic FE, noise dropped']
cols = ['black', '#D55E00', 'grey', '#0072B2']
for j, (s, lab, col) in enumerate(zip(specs, labels, cols)):
    g = res8[res8.model == s].loc[['level_elec2017', 'level_elec2021', 'level_elec2025']]
    yy = np.arange(3)[::-1] + (1.5 - j) * 0.18
    ax.errorbar(g.coef, yy, xerr=[g.coef - g.lo, g.hi - g.coef], fmt='o', color=col, capsize=3, label=lab)
ax.set_yticks(np.arange(3)[::-1]); ax.set_yticklabels(['2017', '2021', '2025']); ax.axvline(0, color='grey', lw=0.8)
ax.set_xlabel('Level shift at the election breakpoint (95% CI)'); ax.legend(frameon=False, fontsize=10); ax.grid(alpha=0.3)
ax.text(0.01, 0.97, 'A', transform=ax.transAxes, fontsize=16, fontweight='bold', va='top')
ax = axes[1]
excl_pp.plot(kind='bar', ax=ax, color=['#999999', '#666666', '#333333'], width=0.8)
ax.set_ylabel('share of speeches in the BERTopic noise cluster'); ax.set_xlabel(''); ax.legend(frameon=False, title='period'); ax.grid(alpha=0.3, axis='y')
ax.text(0.01, 0.97, 'B', transform=ax.transAxes, fontsize=16, fontweight='bold', va='top')
plt.tight_layout(); plt.savefig(os.path.join(OUT8, 'topic_controls_parliament.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(OUT8, 'topic_controls_parliament.png'), dpi=150, bbox_inches='tight'); plt.show()
tw_res = os.path.join(OUT8, 'twitter_same_sample_models.csv')
if os.path.exists(tw_res):
    tr = pd.read_csv(tw_res, index_col=0); NUM8['twitter_models'] = {m: g.coef.round(5).to_dict() for m, g in tr.groupby('model')}
    display(tr[tr.index.isin(['time_centered', 'leaning[T.right]', 'leaning[T.left]', 'time_centered:leaning[T.right]', 'log_like_count', 'emi_lag1'])]
            .reset_index().pivot(index='term', columns='model', values='coef').round(5))
save_numbers(NUM8, OUT8)

**Reading the output.** Panel A shows whether the election level shifts move when topic fixed effects are added on
the *same* sample, with the noise cluster kept as its own category, and again when noise documents are dropped. Panel B
shows that the noise share is flat over time and similar across parties, so the earlier exclusion could not have
created a trend or a party difference. The Supplement text ("documents in the noise cluster were excluded") is to be
replaced by the noise-kept specification as the main robustness check, with the dropped version as a secondary one.

## 9. Who moved? Ideological distance and constituency exposure as predictors of the within-MdB change at 2017

Two mechanisms that would tie the 2017 shift to the AfD rather than to an election-cycle regularity: **affinity**
(MdBs ideologically closer to the AfD adopt its register sooner or more) and **threat** (MdBs facing a stronger AfD in
their own constituency adapt more). Both are tested *within parties* on the same politicians across the 2017 breakpoint.

- Positions: Breunig & Guinaudeau (2025, PSRM), expert pairwise comparisons for all 709 MdBs of the 19th Bundestag,
  matched to `politicianId` in `data/revision/breunig_guinaudeau/mp_positions_wp19.csv`; `dist_afd` = |position − AfD centroid|.
- Exposure: AfD second-vote share 2017 and 2013 in the MdB's own constituency (list MdBs: the constituency they ran
  in), from `code/04_analysis/mechanisms/0434_build_constituency_exposure.py` → `data/revision/mdb_exposure_wp19.csv`.
- Outcome: within-MdB change in mean EMI, six full quarters after the 2017 transition (2018Q1–2019Q2) minus six
  before (2016Q1–2017Q2), at least five speeches on each side; plus a quarterly event study for timing.
- Models: OLS of the change on the predictor with party fixed effects, East/West, direct mandate and gender, clustered
  by Land; the event study adds MdB fixed effects and party × quarter effects, so only within-party gradients remain.
  A null here is informative: it would mean the 2017 drop was not transmitted along ideological proximity or electoral
  threat, which favours a chamber-level mechanism.

In [ ]:
import statsmodels.formula.api as smf
OUT9 = os.path.join(RES, '09_mechanism_2017'); os.makedirs(OUT9, exist_ok=True)
NUM9 = {}
pos = pd.read_csv(os.path.join(HERE, 'data', 'revision', 'breunig_guinaudeau', 'mp_positions_wp19.csv'), dtype={'politicianId': str})
exp = pd.read_csv(os.path.join(HERE, 'data', 'revision', 'mdb_exposure_wp19.csv'), dtype={'politicianId': str})
sp = par.copy(); sp['politicianId'] = sp.politicianId.astype(str)
sp['q'] = sp.date.dt.tz_localize(None).dt.to_period('Q')
PRE = (pd.Period('2016Q1'), pd.Period('2017Q2')); POST = (pd.Period('2018Q1'), pd.Period('2019Q2'))
pre = sp[(sp.q >= PRE[0]) & (sp.q <= PRE[1])].groupby('politicianId')[EMI].agg(['mean', 'size']).rename(columns={'mean': 'emi_pre', 'size': 'n_pre'})
post = sp[(sp.q >= POST[0]) & (sp.q <= POST[1])].groupby('politicianId')[EMI].agg(['mean', 'size']).rename(columns={'mean': 'emi_post', 'size': 'n_post'})
ch = pre.join(post, how='inner'); ch = ch[(ch.n_pre >= 5) & (ch.n_post >= 5)]; ch['change'] = ch.emi_post - ch.emi_pre
m9 = (ch.reset_index().merge(pos[['politicianId', 'party', 'position', 'dist_afd', 'ci_low', 'ci_high', 'faction', 'gender']], on='politicianId', how='inner')
      .merge(exp[['politicianId', 'east', 'direct', 'land', 'afd_share_2017', 'afd_share_2013', 'afd_change', 'own_share_2017', 'afd_minus_own', 'AfD_land_2017']], on='politicianId', how='left'))
m9 = m9[m9.party != 'AfD'].copy()                      # AfD MdBs have no pre-2017 speeches anyway
for c in ['dist_afd', 'afd_share_2017', 'afd_change', 'afd_minus_own']:
    m9[f'{c}_z'] = (m9[c] - m9[c].mean()) / m9[c].std()                           # chamber-wide z
    m9[f'{c}_wz'] = m9.groupby('party')[c].transform(lambda s: (s - s.mean()) / s.std())   # within-party z
NUM9['n_panel'] = int(len(m9)); NUM9['by_party'] = m9.groupby('party').size().to_dict()
print('panel MdBs with >=5 speeches before and after 2017 and an expert position:', len(m9)); print(m9.groupby('party').size().to_dict())
print('mean change by party:'); print(m9.groupby('party').change.agg(['mean', 'std', 'size']).round(3).to_string())
m9.to_csv(os.path.join(OUT9, 'panel_2017.csv'), index=False)

In [ ]:
# --- 9.1 cross-sectional dose-response: change ~ predictor + party FE + controls, clustered by Land ---------
def ols(formula, d, tag):
    model = smf.ols(formula, data=d)                                     # rows with missing values are dropped here
    g = pd.factorize(d.land.fillna('NA').loc[model.data.row_labels])[0]   # cluster ids aligned on the rows used
    return model.fit(cov_type='cluster', cov_kwds={'groups': g})
specs = {
    'distance (chamber z)':      'change ~ dist_afd_z + C(party) + east + direct + C(gender)',
    'distance (within-party z)': 'change ~ dist_afd_wz + C(party) + east + direct + C(gender)',
    'AfD share 2017 (z)':        'change ~ afd_share_2017_z + C(party) + east + direct + C(gender)',
    'AfD change 2013-17 (z)':    'change ~ afd_change_z + C(party) + east + direct + C(gender)',
    'AfD minus own share (z)':   'change ~ afd_minus_own_z + C(party) + east + direct + C(gender)',
    'distance + AfD share':      'change ~ dist_afd_z + afd_share_2017_z + C(party) + east + direct + C(gender)',
    'distance x party':          'change ~ dist_afd_wz * C(party) + east + direct + C(gender)',
}
rows = []
for tag, f in specs.items():
    r = ols(f, m9.dropna(subset=['afd_share_2017', 'dist_afd']), tag)
    for term in r.params.index:
        if term.startswith(('dist_afd', 'afd_')) or ':' in term:
            rows.append(dict(spec=tag, term=term, coef=r.params[term], se=r.bse[term], p=r.pvalues[term], lo=r.conf_int().loc[term, 0], hi=r.conf_int().loc[term, 1], n=int(r.nobs), r2=r.rsquared))
res9 = pd.DataFrame(rows); res9.to_csv(os.path.join(OUT9, 'dose_response_models.csv'), index=False)
NUM9['dose_response'] = {f'{r.spec}|{r.term}': dict(coef=r.coef, se=r.se, p=r.p, n=r.n) for r in res9.itertuples()}
pd.set_option('display.width', 200); res9[~res9.term.str.contains(':')].round(4)

In [ ]:
# --- 9.2 figure: within-MdB change vs distance and vs exposure, by party, with within-party fits --------------
PC = {'CDU/CSU': '#000000', 'SPD': '#E3000F', 'FDP': '#D4B300', 'Greens': '#46962B', 'Left': '#BE3075'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, xcol, xlab in [(axes[0], 'dist_afd', 'ideological distance to the AfD centroid (Breunig & Guinaudeau)'),
                       (axes[1], 'afd_share_2017', 'AfD second-vote share 2017 in own constituency')]:
    for pty, d in m9.dropna(subset=[xcol]).groupby('party'):
        ax.scatter(d[xcol], d.change, s=14, alpha=0.45, color=PC.get(pty, 'grey'), label=f'{pty} (n={len(d)})')
        if len(d) >= 15:
            b = np.polyfit(d[xcol], d.change, 1); xs = np.linspace(d[xcol].min(), d[xcol].max(), 20)
            ax.plot(xs, np.polyval(b, xs), color=PC.get(pty, 'grey'), lw=2)
    ax.axhline(0, color='grey', lw=0.8); ax.set_xlabel(xlab); ax.set_ylabel('within-MdB change in EMI (2018Q1–2019Q2 minus 2016Q1–2017Q2)'); ax.grid(alpha=0.3); ax.legend(frameon=False, fontsize=9)
axes[0].text(0.01, 0.97, 'A', transform=axes[0].transAxes, fontsize=16, fontweight='bold', va='top')
axes[1].text(0.01, 0.97, 'B', transform=axes[1].transAxes, fontsize=16, fontweight='bold', va='top')
plt.tight_layout(); plt.savefig(os.path.join(OUT9, 'change_vs_distance_exposure.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(OUT9, 'change_vs_distance_exposure.png'), dpi=150, bbox_inches='tight'); plt.show()

### 9.3 Event study: does the gradient appear at the breakpoint, and does it persist?

Quarterly MdB means 2015Q1–2019Q4 for the same panel. Model: EMI_it = α_i + γ_party(i),t + Σ_t β_t · x_i · 1[quarter = t] + ε_it,
with x the within-party standardised predictor and 2017Q2 as the reference quarter. Party × quarter effects absorb every
party-level trend and the chamber-wide drop, so β_t traces only whether, inside parties, MdBs closer to the AfD (or more
exposed) sit lower than their colleagues in each quarter relative to just before the election.

In [ ]:
ev = sp[sp.politicianId.isin(m9.politicianId) & (sp.q >= pd.Period('2015Q1')) & (sp.q <= pd.Period('2019Q4'))]
ev = ev.groupby(['politicianId', 'q'])[EMI].agg(['mean', 'size']).reset_index().rename(columns={'mean': 'y', 'size': 'n'})
ev = ev[ev.n >= 2].merge(m9[['politicianId', 'party', 'dist_afd_wz', 'afd_share_2017_wz']], on='politicianId')
ev['qs'] = ev.q.astype(str); REF = '2017Q2'
quarters = sorted(ev.qs.unique())
def event_study(xcol):
    d = ev.dropna(subset=[xcol]).copy()
    for qq in quarters:
        if qq != REF: d[f'x_{qq}'] = d[xcol] * (d.qs == qq)
    f = 'y ~ C(politicianId) + C(party):C(qs) + ' + ' + '.join(f'x_{qq}' for qq in quarters if qq != REF)
    model = smf.ols(f, data=d); r = model.fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(d.politicianId.loc[model.data.row_labels])[0]})
    out = pd.DataFrame({'quarter': [qq for qq in quarters if qq != REF],
                        'coef': [r.params[f'x_{qq}'] for qq in quarters if qq != REF],
                        'lo': [r.conf_int().loc[f'x_{qq}', 0] for qq in quarters if qq != REF],
                        'hi': [r.conf_int().loc[f'x_{qq}', 1] for qq in quarters if qq != REF]})
    # pooled post-period effect for the numbers file
    d['post'] = (d.q > pd.Period('2017Q3')).astype(int); d['x_post'] = d[xcol] * d.post
    mp_ = smf.ols('y ~ C(politicianId) + C(party):C(qs) + x_post', data=d); rp = mp_.fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(d.politicianId.loc[mp_.data.row_labels])[0]})
    return out, dict(coef=float(rp.params['x_post']), se=float(rp.bse['x_post']), p=float(rp.pvalues['x_post']), n_obs=int(rp.nobs), n_mdb=int(d.politicianId.nunique()))
es_dist, pooled_dist = event_study('dist_afd_wz'); es_exp, pooled_exp = event_study('afd_share_2017_wz')
NUM9['event_study_post_x_predictor'] = {'distance_within_party_z': pooled_dist, 'afd_share_within_party_z': pooled_exp}
es_dist.to_csv(os.path.join(OUT9, 'event_study_distance.csv'), index=False); es_exp.to_csv(os.path.join(OUT9, 'event_study_exposure.csv'), index=False)
print('post x distance (within-party z):', {k: round(v, 4) if isinstance(v, float) else v for k, v in pooled_dist.items()})
print('post x AfD share (within-party z):', {k: round(v, 4) if isinstance(v, float) else v for k, v in pooled_exp.items()})
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, es, lab in [(axes[0], es_dist, 'distance to AfD (within-party z)'), (axes[1], es_exp, 'AfD constituency share (within-party z)')]:
    x = pd.PeriodIndex(es.quarter, freq='Q').to_timestamp()
    ax.errorbar(x, es.coef, yerr=[es.coef - es.lo, es.hi - es.coef], fmt='o-', color='black', capsize=2, ms=4)
    ax.axhline(0, color='grey', lw=0.8); ax.axvline(pd.Timestamp('2017-07-01'), color='#D55E00', ls=':', lw=1.4)
    ax.set_title(f'β_t for {lab}, ref. 2017Q2'); ax.grid(alpha=0.3); ax.set_xlabel('quarter')
axes[0].set_ylabel('effect of one within-party SD on quarterly EMI')
plt.tight_layout(); plt.savefig(os.path.join(OUT9, 'event_study.pdf'), bbox_inches='tight'); plt.savefig(os.path.join(OUT9, 'event_study.png'), dpi=150, bbox_inches='tight'); plt.show()
save_numbers(NUM9, OUT9)

**Reading the output.** In 9.1 a negative, significant coefficient on distance means closer MdBs dropped *less*
(distance is measured from the AfD, so affinity predicts a *positive* coefficient: further away, smaller drop, i.e. less
negative change). For exposure, threat predicts a *negative* coefficient. The event study shows whether any gradient
opens exactly at the breakpoint and whether it persists or closes; a flat line through zero after 2017 means the drop
was uniform within parties along that dimension.

## 10. Mechanism battery (all remaining tests of the contagion hypothesis)

Run by `code/04_analysis/mechanisms/0431_mechanism_tests.py`, results in `results/revision/10_mechanism_tests/`. Tests: **T3** 2021 transition on the stayer panel
with the 2017 positions carried forward; **T4** discrete-time hazard of "adoption" (first quarter after 2017 in which an
MdB's two-quarter EMI falls half a within-person SD below their pre-2017 mean); **T5** long-run drift 2015–2025, distance
× time with MdB and party × quarter effects; **T7** similarity of each MdB's speech embedding to the AfD centroid, change
across 2017, and its relation to the EMI change; **T8** contact with the AfD in debates (share of speeches right after an
AfD speaker; share in agenda items where the AfD spoke) and contact × distance; **T9** intra-party wing codes; **T10**
placebo at the 2013 transition on the re-scored 2000–2025 series; **T12** lead–lag with the AfD's quarterly EMI.
Skipped: T6 (identical to T5 under quarter effects), T11 (needs the attribution coding), T13 (no Twitter–MdB map).

In [ ]:
import runpy
OUT10 = os.path.join(RES, '10_mechanism_tests')
if not os.path.exists(os.path.join(OUT10, 'tests_summary.csv')):
    runpy.run_path(os.path.join(HERE, 'code/04_analysis/mechanisms/0431_mechanism_tests.py'))
summ = pd.read_csv(os.path.join(OUT10, 'tests_summary.csv'))
pd.set_option('display.width', 220)
display(summ[['test', 'outcome', 'predictor', 'coef', 'se', 'p', 'n', 'sig05']].round(4))
print('flagged at p < 0.05:'); display(summ[summ.sig05][['test', 'outcome', 'predictor', 'coef', 'se', 'p', 'n', 'note']].round(4))
from IPython.display import Image, display as _d
if os.path.exists(os.path.join(OUT10, 'T7_emi_vs_similarity.png')): _d(Image(os.path.join(OUT10, 'T7_emi_vs_similarity.png')))

**Reading the output.** Three coefficients clear p < 0.05 out of 37: the within-party association between an MdB's
move towards the AfD's language and their EMI drop (T7, strong), the interaction of same-agenda-item contact with
distance (T8, one of four contact tests, with little variance in the contact share), and the SPD "Netzwerk Berlin" wing
(T9, six MdBs). Every direct test of the contagion gradient (distance or exposure predicting extent, speed, drift,
similarity change, or lead–lag) is null, at 2017 and at 2021, and so is the 2013 placebo.

## 12. Does the discourse become more intuition-based right after the AfD speaks?

Within a sitting, speeches are ordered. For every AfD speech we compare the EMI of the k non-AfD speeches immediately
before and immediately after it (same sitting), and do the same around every other party's speeches as placebo. Checks:
position in the agenda item (item fixed effects and position dummies), local debate climate (intuition-heavy speeches
of other parties as placebo), the distance profile, and whether the following speech addresses the AfD by name.
Finally the paper's breakpoint model is re-estimated with a 'mentions the AfD' control and on speeches that never
mention the AfD, to size the confrontation component of the 2017 shift. Scripts: `code/04_analysis/contagion/0415_afd_sequence_effect.py`,
`code/04_analysis/contagion/0417_afd_local_window.py`, `code/04_analysis/contagion/0418_afd_local_window_checks.py`, `code/04_analysis/contagion/0419_afd_confrontation.py`; outputs in `results/revision/12_afd_sequence/`.

In [ ]:
OUT12 = os.path.join(RES, '12_afd_sequence')
from IPython.display import Image, display as _d
R = pd.read_csv(os.path.join(OUT12, 'local_window_results.csv'))
print('after minus before (k speeches), mean EMI of other parties, by party of the speech:')
display(R[R.variant == 'all speeches'].pivot(index='k', columns='party', values='diff').round(3))
display(R[R.variant != 'all speeches'][['party', 'variant', 'diff', 'se', 'p', 'n_speeches']].round(3))
_d(Image(os.path.join(OUT12, 'local_window.png')))
print('party-mention placebo (speaker FE, non-members):'); display(pd.read_csv(os.path.join(OUT12, 'confrontation_mentions_by_party.csv')).round(3))
qq = pd.read_csv(os.path.join(OUT12, 'confrontation_quarterly.csv'), index_col=0); display(qq.round(3).iloc[::4])
_d(Image(os.path.join(OUT12, 'confrontation.png')))
NUM12 = json.load(open(os.path.join(OUT12, 'numbers.json'))) if os.path.exists(os.path.join(OUT12, 'numbers.json')) else {}
NUM12

**Reading the output.** The local drop after AfD speeches is AfD-specific (no other party produces it), survives
position and item fixed effects, scales with how intuition-based the AfD speech was, and is confined to replies that
name the AfD; replies that do not engage the AfD show no drop. Mentioning the AfD lowers a speaker's EMI by about 0.5
within person, far more than mentioning any other party. With a mention control, the 2017 level shift falls from about
−0.44 to −0.31, and on speeches that never mention the AfD it is −0.35: roughly a quarter to a third of the 2017 shift is
confrontation with the AfD, the remainder is a shift in speeches that do not engage the AfD at all.